# 03 · Validación cruzada

**Tema 3 del proyecto.** Se mide qué tan bien generaliza cada uno de los 5 modelos con los
hiperparámetros que eligió Hyperband (notebook 02).

1. **K-fold estratificado (k = 5)** con los 5 modelos: método principal.
2. Comparación de métodos con la CNN propia: **K-fold sin estratificar, Monte Carlo
   (shuffle-split) y hold-out**.
3. Por qué **leave-one-out, leave-p-out y series de tiempo** no aplican a este caso.

Todo se hace sobre el subconjunto de train (10.000 imágenes, 2.000 por clase). Test no se toca.
Cada modelo entrena el número de épocas que le asignó Hyperband, **sin early stopping**: el fold
de validación solo se usa para medir, nunca para decidir cuándo parar.

**Requisitos de la sesión:** GPU T4 x2, Internet On y, como Input, el dataset y las salidas de
`01-preparacion-datos` y `02_hyperband`.

**Salida:** `cv_folds.csv`, `cv_resumen.csv`, `cv_f1_por_clase.csv`, `cv_metodos.csv` y
`oof_<modelo>.npy` (probabilidades fuera de fold, que usa el ensamble híbrido del tema 5).

### Modo de prueba

Con `QUICK = True` todo corre en miniatura (un fold, 1 época, pocas imágenes) para detectar
errores en pocos minutos. **Para la corrida real ponlo en `False` antes de Save & Run All.**

In [ ]:
QUICK = True

In [ ]:
import os

# Asignador de memoria que devuelve la GPU al soltar cada modelo (evita fragmentación).
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

import gc
import sys
import time
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.model_selection import KFold, ShuffleSplit, StratifiedShuffleSplit

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


SPLITS = find_dir_with("subset_train.csv", "rice_data.py")
HB = find_dir_with("best_hparams.csv", "rice_models.py")
sys.path += [str(SPLITS), str(HB)]

from rice_data import CLASSES, SEED, find_data_root, make_dataset
from rice_models import MODEL_NAMES, build_model

keras.mixed_precision.set_global_policy("mixed_float16")
DATA_ROOT = find_data_root()
print("Particiones:", SPLITS)
print("Hyperband:", HB)

## 1. Datos e hiperparámetros

In [ ]:
K = 5
sub_train = pd.read_csv(SPLITS / "subset_train.csv")
if QUICK:
    sub_train = sub_train.groupby("fold", group_keys=False).sample(n=60, random_state=SEED)
sub_train = sub_train.reset_index(drop=True)

best = pd.read_csv(HB / "best_hparams.csv").set_index("model")
best["epochs_cv"] = 1 if QUICK else best["epochs"].fillna(5).astype(int)
FOLDS = [0] if QUICK else list(range(K))

print("Imágenes:", len(sub_train), "| folds a ejecutar:", FOLDS)
best[["lr", "dropout", "batch_size", "epochs_cv"]]

In [ ]:
def train_eval(name, train_df, val_df):
    """Entrena un modelo con sus hiperparámetros de Hyperband y devuelve sus probabilidades en val."""
    hp = best.loc[name]
    keras.backend.clear_session()
    gc.collect()
    keras.utils.set_random_seed(SEED)
    model = build_model(name, lr=float(hp["lr"]), dropout=float(hp["dropout"]))

    t0 = time.time()
    model.fit(make_dataset(train_df, DATA_ROOT, batch_size=int(hp["batch_size"]), training=True),
              epochs=int(hp["epochs_cv"]), verbose=0)
    t_train = time.time() - t0

    t0 = time.time()
    probs = model.predict(make_dataset(val_df, DATA_ROOT, batch_size=64), verbose=0)
    ms_img = (time.time() - t0) / len(val_df) * 1000

    params = model.count_params()
    del model
    return np.asarray(probs, dtype="float32"), t_train, ms_img, params


def metricas(y, probs):
    probs = probs.astype("float64")
    probs = probs / probs.sum(axis=1, keepdims=True)
    pred = probs.argmax(1)
    return {
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
        "precision_macro": precision_score(y, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, pred, average="macro", zero_division=0),
        "log_loss": log_loss(y, probs, labels=list(range(len(CLASSES)))),
    }

## 2. K-fold estratificado con los 5 modelos

El subconjunto se dividió en 5 folds en el notebook 01, cada uno con 400 imágenes por clase.
En cada vuelta un fold se usa como validación y los otros 4 para entrenar, así cada imagen se
evalúa exactamente una vez con un modelo que no la vio.

Las predicciones de cada vuelta se guardan (*out-of-fold*). Con ellas se arman la matriz de
confusión y el ensamble híbrido del tema 5 sin fuga de datos.

Si un modelo falla en un fold, el error queda registrado y la corrida continúa con el siguiente.

In [ ]:
filas = []
t_total = time.time()
for name in MODEL_NAMES:
    oof = np.full((len(sub_train), len(CLASSES)), np.nan, dtype="float32")
    for fold in FOLDS:
        tr = sub_train[sub_train["fold"] != fold]
        va = sub_train[sub_train["fold"] == fold]
        try:
            probs, t_train, ms_img, params = train_eval(name, tr, va)
        except Exception as e:
            print(f"ERROR {name} fold {fold}: {e!r}")
            filas.append({"model": name, "fold": fold, "error": repr(e)[:300]})
            continue
        oof[va.index] = probs
        m = metricas(va["label_idx"].to_numpy(), probs)
        filas.append({"model": name, "fold": fold, **m, "train_s": t_train,
                      "infer_ms_img": ms_img, "params_M": params / 1e6})
        print(f"{name:15s} fold {fold}  acc={m['accuracy']:.4f}  f1={m['f1_macro']:.4f}  "
              f"train={t_train / 60:.1f} min")
    np.save(OUT_DIR / f"oof_{name}.npy", oof)
    pd.DataFrame(filas).to_csv(OUT_DIR / "cv_folds.csv", index=False)

print(f"K-fold terminado en {(time.time() - t_total) / 60:.0f} min")
folds = pd.DataFrame(filas)
folds

### Resumen por modelo (media ± desviación entre folds)

Como este dataset es fácil y casi todos los modelos superan 99 %, la tabla también compara
log-loss (qué tan seguros están de sus aciertos), tiempo de entrenamiento, tiempo de inferencia
por imagen y tamaño del modelo.

In [ ]:
ok = folds.dropna(subset=["accuracy"])
resumen = ok.groupby("model").agg(
    accuracy_media=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    f1_media=("f1_macro", "mean"), f1_std=("f1_macro", "std"),
    log_loss_media=("log_loss", "mean"),
    train_min_por_fold=("train_s", lambda s: s.mean() / 60),
    infer_ms_img=("infer_ms_img", "mean"),
    params_M=("params_M", "first"),
    folds_ok=("fold", "count"),
).reindex([m for m in MODEL_NAMES if m in set(ok["model"])])
resumen.to_csv(OUT_DIR / "cv_resumen.csv")
resumen.sort_values("log_loss_media").round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
datos = [ok.loc[ok["model"] == m, "accuracy"] for m in resumen.index]
axes[0].boxplot(datos)
axes[0].set_xticks(range(1, len(datos) + 1), resumen.index, rotation=20)
axes[0].set_ylabel("Accuracy por fold")
axes[0].set_title("Variación entre folds")

axes[1].scatter(resumen["params_M"], resumen["accuracy_media"], s=80)
for m, r in resumen.iterrows():
    axes[1].annotate(m, (r["params_M"], r["accuracy_media"]), textcoords="offset points", xytext=(6, 4))
axes[1].set_xscale("log")
axes[1].set_xlabel("Parámetros (millones, escala log)")
axes[1].set_ylabel("Accuracy media")
axes[1].set_title("Exactitud frente a tamaño del modelo")
plt.tight_layout()
plt.show()

### Matrices de confusión y F1 por clase (predicciones *out-of-fold*)

Muestran **qué variedades se confunden entre sí**, que es lo que distingue a los modelos cuando
la exactitud global es casi igual.

In [ ]:
y_all = sub_train["label_idx"].to_numpy()
modelos_ok = list(resumen.index)
fig, axes = plt.subplots(1, len(modelos_ok), figsize=(4.2 * len(modelos_ok), 4.2), squeeze=False)
f1_clase = {}
for ax, name in zip(axes[0], modelos_ok):
    oof = np.load(OUT_DIR / f"oof_{name}.npy")
    mask = ~np.isnan(oof).any(axis=1)
    pred = oof[mask].argmax(1)
    ConfusionMatrixDisplay.from_predictions(
        y_all[mask], pred, labels=list(range(len(CLASSES))), display_labels=CLASSES,
        ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(name)
    rep = classification_report(y_all[mask], pred, labels=list(range(len(CLASSES))),
                                target_names=CLASSES, output_dict=True, zero_division=0)
    f1_clase[name] = {c: rep[c]["f1-score"] for c in CLASSES}
plt.tight_layout()
plt.show()

f1_clase = pd.DataFrame(f1_clase).T
f1_clase.to_csv(OUT_DIR / "cv_f1_por_clase.csv")
f1_clase.round(4)

## 3. Comparación de métodos de validación cruzada

Para comparar métodos sin gastar horas de GPU se usa solo la **CNN propia**, el modelo más
rápido. Mismo modelo y mismos hiperparámetros; lo único que cambia es cómo se parten los datos.

| Método | Cómo parte los datos |
|---|---|
| K-fold estratificado | 5 folds con la misma proporción de clases (resultado de la sección 2) |
| K-fold | 5 folds al azar, sin controlar la proporción de clases |
| Monte Carlo (shuffle-split) | 5 particiones aleatorias independientes 80/20; una imagen puede validar varias veces o ninguna |
| Hold-out | Una sola partición estratificada 80/20 |

Primero, sin entrenar nada: cuánto se desbalancean las clases dentro de cada fold según el
método.

In [ ]:
def desbalance(splitter, y):
    """Mayor desviación (en puntos porcentuales) de la proporción de una clase en un fold de validación."""
    esperado = np.bincount(y, minlength=len(CLASSES)) / len(y)
    peor = 0
    for _, va in splitter:
        prop = np.bincount(y[va], minlength=len(CLASSES)) / len(va)
        peor = max(peor, np.abs(prop - esperado).max() * 100)
    return peor

y_full = pd.read_csv(SPLITS / "subset_train.csv")["label_idx"].to_numpy()
idx = np.zeros(len(y_full))
estrat = pd.read_csv(SPLITS / "subset_train.csv")["fold"].to_numpy()
pd.DataFrame({
    "método": ["K-fold estratificado", "K-fold", "Monte Carlo", "Hold-out (estratificado)"],
    "máx. desviación de proporción (p.p.)": [
        desbalance(((None, np.where(estrat == f)[0]) for f in range(K)), y_full),
        desbalance(KFold(K, shuffle=True, random_state=SEED).split(idx), y_full),
        desbalance(ShuffleSplit(K, test_size=0.2, random_state=SEED).split(idx), y_full),
        desbalance(StratifiedShuffleSplit(1, test_size=0.2, random_state=SEED).split(idx, y_full), y_full),
    ],
}).round(3)

In [ ]:
METODO_MODELO = "cnn_propia"
y = sub_train["label_idx"].to_numpy()
idx = np.zeros(len(sub_train))
n_rep = 1 if QUICK else K

metodos = {
    "K-fold": KFold(K, shuffle=True, random_state=SEED).split(idx),
    "Monte Carlo": ShuffleSplit(n_rep, test_size=0.2, random_state=SEED).split(idx),
    "Hold-out": StratifiedShuffleSplit(1, test_size=0.2, random_state=SEED).split(idx, y),
}

filas_m = [{"método": "K-fold estratificado", "split": int(r["fold"]), "accuracy": r["accuracy"],
            "f1_macro": r["f1_macro"]}
           for _, r in ok[ok["model"] == METODO_MODELO].iterrows()]
for metodo, splits in metodos.items():
    for i, (tr, va) in enumerate(splits):
        if i >= n_rep:
            break
        probs, *_ = train_eval(METODO_MODELO, sub_train.iloc[tr], sub_train.iloc[va])
        m = metricas(y[va], probs)
        filas_m.append({"método": metodo, "split": i, "accuracy": m["accuracy"], "f1_macro": m["f1_macro"]})
        print(f"{metodo:12s} split {i}  acc={m['accuracy']:.4f}")

cv_metodos = pd.DataFrame(filas_m)
cv_metodos.to_csv(OUT_DIR / "cv_metodos.csv", index=False)
cv_metodos.groupby("método")["accuracy"].agg(["mean", "std", "count"]).round(4)

**Cómo leer la tabla:**

- **Media:** si los métodos dan medias parecidas, la estimación del desempeño es confiable.
- **Desviación estándar:** mide qué tan estable es la estimación. Hold-out tiene una sola
  medición, así que no puede estimar su propia variabilidad; esa es su principal debilidad.
- **Estratificar** importa más cuando hay clases desbalanceadas o pocas muestras. Aquí el
  subconjunto está balanceado, por eso la diferencia con K-fold simple debería ser pequeña,
  aunque la tabla de desbalance anterior muestra que la estratificación lo elimina por completo.

## 4. Métodos que no aplican a este caso

| Método | Por qué no se usa aquí |
|---|---|
| Leave-one-out | Exige un entrenamiento por imagen: 10.000 entrenamientos del subconjunto. Con la CNN propia (la más rápida) serían semanas de GPU. Se usa con datasets de decenas o pocos cientos de muestras. |
| Leave-p-out | Entrena con todas las combinaciones posibles de *p* imágenes de validación. Con p = 2 son C(10.000, 2) ≈ 50 millones de entrenamientos: inviable. |
| Series de tiempo (rolling) | Sirve cuando los datos tienen orden temporal y no se puede validar con el pasado usando el futuro. Las imágenes de granos de arroz no tienen orden en el tiempo. |